# P106 — OSWorld: evaluación de agentes multimodales en tareas abiertas sobre entornos informáticos reales

## 1. Título y paper

**Paper:** *OSWorld: Benchmarking Multimodal Agents for Open-Ended Tasks in Real Computer Environments*  
**Autoría:** Tianbao Xie, Danyang Zhang, Jixuan Chen, Xiaochuan Li, Siheng Zhao, y otros  
**Año y venue:** 2024 · NeurIPS 2024 · arXiv:2404.07972  
**Nivel:** L3 · **Motor:** `osworld`  
**Ficha completa:** [`P106_osworld`](../../papers/foundational/P106_osworld/README.md)

**Hito:** Lleva la evaluación de agentes al escritorio completo, con tareas que cruzan aplicaciones y un verificador por tarea que inspecciona el sistema real.

- [arXiv:2404.07972](https://arxiv.org/abs/2404.07972)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los bancos de pruebas de agentes se limitaban al navegador o a entornos de juguete. El trabajo de oficina real cruza aplicaciones —hoja de cálculo, ficheros, terminal, navegador— y ahí no había forma comparable de medir nada.
2. Ejecutar una implementación mínima de la propuesta: Un entorno de escritorio completo en máquina virtual con estado reiniciable, cientos de tareas reales recogidas de usuarios, y para cada una un script de verificación que inspecciona el estado final del sistema: una celda, un fichero, un código de salida.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P104
- P105
- P51


## 4. Intuición

El trabajo de oficina real cruza aplicaciones: descargar algo, abrirlo en una hoja de cálculo, ejecutar un script y guardar el resultado donde toca. OSWorld pone al agente en un escritorio completo y comprueba el resultado inspeccionando el sistema, no preguntando.


## 5. Concepto mínimo

```text
Máquina virtual con estado reiniciable
+ tareas reales recogidas de usuarios
+ UN VERIFICADOR POR TAREA que inspecciona el estado final:
      leer la celda B7 · comprobar que el fichero existe · código de salida

Nadie le pregunta al agente si lo consiguió.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('osworld', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué tasa consiguen las personas? ¿Y el agente?
2. ¿Dónde está la diferencia: en tareas de una aplicación o de varias?
3. ¿Qué hace comparables los resultados entre trabajos?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('osworld', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('osworld', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Las personas resuelven **8 de 8** y el agente **3 de 8**. Con una sola aplicación acierta 3 de 6; con varias, **0 de 2**. Lo que rompe no es la dificultad de cada paso: es mantener el objetivo mientras se cambia de contexto.


## 10. Comentario pedagógico

El verificador por tarea es lo que hace honesta la evaluación y también lo que la limita: escribir uno por tarea es caro, así que el banco de pruebas tiene cientos de tareas y no millones. Es el mismo compromiso que en [SWE-bench](../../papers/foundational/P51_swebench/README.md): la verificación ejecutable cuesta, y es lo único que impide que un agente elocuente puntúe alto.


## 11. Error o anti-patrón deliberado

Anti-patrón: citar una tasa de éxito de agentes sin decir cómo se verificó.


In [ ]:
print('«El agente resuelve el 40% de las tareas» no significa nada sin el protocolo.')
print('Verificado por estado final, por juicio de otro modelo o por su propio informe')
print('son tres numeros distintos, y el orden entre ellos es siempre el mismo.')

## 12. Corrección

El protocolo que hace comparable el número:


In [ ]:
r = run_paper_lab('osworld', seed=7)['result']
print('humanos:', r['tasa_humana'], '| agente:', r['tasa_del_agente'])
print('una app  :', r['una_sola_aplicacion'])
print('varias   :', r['varias_aplicaciones'])
print()
for v in r['verificadores'][:4]:
    print('  verificador:', v)

## 13. Desafío guiado

Explica por qué las tareas multiaplicación fallan más, y qué capacidad concreta les falta a los agentes ahí.


In [ ]:
r = run_paper_lab('osworld', seed=3)['result']
show(r)

## 14. Desafío autónomo

Define dos tareas de escritorio de tu trabajo y escribe su verificador programático: qué inspeccionar al final para saber si se hizo. Comprueba cuánto cuesta escribirlo.


## 15. Evidencia de aprendizaje

Guarda la comparación humano-agente por tipo de tarea y tu verificador escrito para una tarea propia.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P106_osworld/README.md) · evaluación formal: [`assessments/papers/P106_osworld.md`](../../assessments/papers/P106_osworld.md)


## 16. Cierre

Aquí termina la ruta encarnada. Lo que falta no es capacidad del modelo: es la ingeniería que sostiene un sistema en producción — versionar, observar, detectar deriva y poder volver atrás.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
